# TontinePilot — Few-Shot Declaration Parsing on Amazon Bedrock

Same lab method as class — setup, `query_model` over the Converse API, zero/one/few-shot, temperature, then Prompt Management — but applied to our production model (**Claude Haiku 4.5**, the one running TontinePilot) and our production task: turning *"Awa a payé 20000"* into a structured payment. The final cell invokes the saved prompt **by ARN**, exactly how the Lambda consumes it.

In [1]:
# Amazon Bedrock is fully managed and serverless: no endpoint to create.
import boto3, json

REGION = "us-east-1"
MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"  # our production model
bedrock = boto3.client("bedrock-runtime", region_name=REGION)
bedrock_agent = boto3.client("bedrock-agent", region_name=REGION)  # Prompt Management APIs live here
print("ready:", MODEL_ID)

ready: us.anthropic.claude-haiku-4-5-20251001-v1:0


## Create a query function

One helper for every call below: system prompt + user message through the Converse API, printing input, output and token usage. (Haiku answers directly — no reasoning blocks to separate, unlike DeepSeek-R1.)

In [2]:
def query_model(system, prompt, temperature=0.2, max_tokens=800):
    """Send a system + user prompt to Claude Haiku 4.5 via Converse and print the answer.

    Returns the raw text answer, or None on error.
    """
    print(f"\n--> Sending to model: {MODEL_ID}")
    print("----------------------------------------------------")
    print(f"[System]:\n{system}")
    print(f"[Input Prompt]:\n{prompt}")
    print("----------------------------------------------------")
    print("Waiting for response...")
    try:
        response = bedrock.converse(
            modelId=MODEL_ID,
            system=[{"text": system}],
            messages=[{"role": "user", "content": [{"text": prompt}]}],
            inferenceConfig={"maxTokens": max_tokens, "temperature": temperature},
        )
    except Exception as e:
        print(f"\n[Error] calling Bedrock: {e}")
        return None
    text = response["output"]["message"]["content"][0]["text"]
    print("\n<-- Received response:")
    print("====================================================")
    print(text.strip())
    print("====================================================")
    print("usage:", response.get("usage"))
    return text

# Prompt engineering techniques

Our task contract (shared by every test below): return ONLY JSON
`{"memberId", "memberName", "amount", "recipientName", "confidence"}`.
Unknown payers must come back **empty** — never assigned to another member.

## Zero-shot prompting

Ask without examples — this is the production baseline.

In [3]:
SYSTEM = """You parse informal tontine payment declarations (French, English, Wolof-inflected French).
Return ONLY valid JSON, no markdown: {"memberId": "<best match id or null>", "memberName": "<matched name or empty string>", "amount": <integer>, "recipientName": "<matched name or empty string>", "confidence": <0..1>}
Rules: extract first plausible amount ("20k","vingt mille","20000" -> 20000); confidence 0.95 exact amount+name, 0.8 partial, 0.5 guess.
STRICT: match names ONLY against the known members list. If no named person matches any member, return memberId null, memberName "" and confidence <= 0.4 — never invent or substitute another member. When the declaration names a known member as beneficiary ("I paid 20000 for Awa") and no other payer is identifiable, that member IS the payer of record. First distinct named member = payer, second distinct named member = recipient."""

MEMBERS = 'Known members: [{"id": "m2", "name": "Moussa Ndiaye"}, {"id": "m4", "name": "Cheikh Fall"}]. Standard contribution 20000 FCFA.'

zero_shot_prompt = MEMBERS + ' Declaration: """Moussa a payé 20000 pour Cheikh"""'
query_model(SYSTEM, zero_shot_prompt)


--> Sending to model: us.anthropic.claude-haiku-4-5-20251001-v1:0
----------------------------------------------------
[System]:
You parse informal tontine payment declarations (French, English, Wolof-inflected French).
Return ONLY valid JSON, no markdown: {"memberId": "<best match id or null>", "memberName": "<matched name or empty string>", "amount": <integer>, "recipientName": "<matched name or empty string>", "confidence": <0..1>}
Rules: extract first plausible amount ("20k","vingt mille","20000" -> 20000); confidence 0.95 exact amount+name, 0.8 partial, 0.5 guess.
STRICT: match names ONLY against the known members list. If the payer is not a known member, return memberId null, memberName "" and confidence <= 0.4 — never invent or substitute another member.
[Input Prompt]:
Known members: [{"id": "m2", "name": "Moussa Ndiaye"}, {"id": "m4", "name": "Cheikh Fall"}]. Standard contribution 20000 FCFA. Declaration: """Moussa a payé 20000 pour Cheikh"""
---------------------------------

'```json\n{"memberId": "m2", "memberName": "Moussa Ndiaye", "amount": 20000, "recipientName": "Cheikh Fall", "confidence": 0.95}\n```'

## One-shot prompting

One example to set the JSON shape and the strictness rule, then the real query.

In [4]:
one_shot_prompt = '''Known members: [{"id": "m3", "name": "Awa Sarr"}]. Standard contribution 20000 FCFA.
Example declaration: """John paid 20000"""
Expected JSON: {"memberId": null, "memberName": "", "amount": 20000, "recipientName": "", "confidence": 0.35}

Now parse — Declaration: """Moussa a payé 20000 pour Cheikh""" '''
query_model(SYSTEM, one_shot_prompt)


--> Sending to model: us.anthropic.claude-haiku-4-5-20251001-v1:0
----------------------------------------------------
[System]:
You parse informal tontine payment declarations (French, English, Wolof-inflected French).
Return ONLY valid JSON, no markdown: {"memberId": "<best match id or null>", "memberName": "<matched name or empty string>", "amount": <integer>, "recipientName": "<matched name or empty string>", "confidence": <0..1>}
Rules: extract first plausible amount ("20k","vingt mille","20000" -> 20000); confidence 0.95 exact amount+name, 0.8 partial, 0.5 guess.
STRICT: match names ONLY against the known members list. If the payer is not a known member, return memberId null, memberName "" and confidence <= 0.4 — never invent or substitute another member.
[Input Prompt]:
Known members: [{"id": "m3", "name": "Awa Sarr"}]. Standard contribution 20000 FCFA.
Example declaration: """John paid 20000"""
Expected JSON: {"memberId": null, "memberName": "", "amount": 20000, "recipientName

'```json\n{"memberId": null, "memberName": "", "amount": 20000, "recipientName": "Cheikh", "confidence": 0.35}\n```'

## Few-shot prompting

Three examples, each teaching one rule: a standard declaration, the stranger rule, an amount in words.

In [5]:
few_shot_prompt = '''Known members: [{"id": "m2", "name": "Moussa Ndiaye"}, {"id": "m3", "name": "Awa Sarr"}, {"id": "m4", "name": "Cheikh Fall"}]. Standard contribution 20000 FCFA.
Example 1 - declaration: """Moussa a payé 20000 pour Cheikh""" -> {"memberId": "m2", "memberName": "Moussa Ndiaye", "amount": 20000, "recipientName": "Cheikh Fall", "confidence": 0.95}
Example 2 - declaration: """John paid 20000""" -> {"memberId": null, "memberName": "", "amount": 20000, "recipientName": "", "confidence": 0.35}
Example 3 - declaration: """Awa a versé vingt mille""" -> {"memberId": "m3", "memberName": "Awa Sarr", "amount": 20000, "recipientName": "", "confidence": 0.9}

Now parse — Declaration: """I paid 20k for Awa""" '''
query_model(SYSTEM, few_shot_prompt)


--> Sending to model: us.anthropic.claude-haiku-4-5-20251001-v1:0
----------------------------------------------------
[System]:
You parse informal tontine payment declarations (French, English, Wolof-inflected French).
Return ONLY valid JSON, no markdown: {"memberId": "<best match id or null>", "memberName": "<matched name or empty string>", "amount": <integer>, "recipientName": "<matched name or empty string>", "confidence": <0..1>}
Rules: extract first plausible amount ("20k","vingt mille","20000" -> 20000); confidence 0.95 exact amount+name, 0.8 partial, 0.5 guess.
STRICT: match names ONLY against the known members list. If the payer is not a known member, return memberId null, memberName "" and confidence <= 0.4 — never invent or substitute another member.
[Input Prompt]:
Known members: [{"id": "m2", "name": "Moussa Ndiaye"}, {"id": "m3", "name": "Awa Sarr"}, {"id": "m4", "name": "Cheikh Fall"}]. Standard contribution 20000 FCFA.
Example 1 - declaration: """Moussa a payé 20000 po

'```json\n{"memberId": null, "memberName": "", "amount": 20000, "recipientName": "Awa Sarr", "confidence": 0.35}\n```'

# Temperature and creativity control

Parsing is extraction, not creation: temperature stays at 0.2 everywhere in this notebook (and in production). Try 0.7 once on the stranger case if you are curious — then set it back.

# Save the winner as a managed prompt

Same helpers as the lab (`bedrock-agent` client), pointed at our content. Variables use double braces: `{{members}}`, `{{standard}}`, `{{recipient}}`, `{{declaration}}` — the Lambda fills exactly these at runtime.

In [9]:
def create_managed_prompt(name, prompt_text, input_variables, model_id=MODEL_ID):
    """Create and version a managed prompt in Amazon Bedrock Prompt Management."""
    prompt = bedrock_agent.create_prompt(
        name=name,
        description="TontinePilot few-shot declaration parser (member, amount, recipient; strict unknown handling).",
        variants=[
            {
                "name": "fewshot-v1",
                "templateType": "TEXT",
                "modelId": model_id,
                "templateConfiguration": {
                    "text": {
                        "text": prompt_text,
                        "inputVariables": [{"name": v} for v in input_variables],
                    }
                },
                "inferenceConfiguration": {"text": {"temperature": 0.2, "maxTokens": 800}},
            }
        ],
        defaultVariant="fewshot-v1",
    )
    version = bedrock_agent.create_prompt_version(promptIdentifier=prompt["id"])
    print(f"Saved managed prompt '{name}' (id: {prompt['id']}, version: {version['version']}).")
    return {"name": name, "id": prompt["id"], "arn": prompt["arn"], "version": version["version"]}


def get_managed_prompt(name):
    """Look up a managed prompt by name and print its stored configuration."""
    matches = [
        p
        for p in bedrock_agent.list_prompts(maxResults=100)["promptSummaries"]
        if p["name"] == name
    ]
    if not matches:
        print(f"No managed prompt named '{name}' was found. Save it first.")
        return None
    prompt = bedrock_agent.get_prompt(promptIdentifier=matches[0]["id"])
    variant = prompt["variants"][0]
    print(f"Name:  {prompt['name']}")
    print(f"Model: {variant['modelId']}")
    print("Template:")
    print(variant["templateConfiguration"]["text"]["text"])
    return prompt

In [ ]:
PROMPT_NAME = "tontine-declaration-parser-lab"
PROMPT_TEXT = SYSTEM + "\n\nFollow these examples exactly:\n" + "\n".join([
    'Example declaration: """Moussa a payé 20000 pour Cheikh""" -> {"memberId": "m2", "memberName": "Moussa Ndiaye", "amount": 20000, "recipientName": "Cheikh Fall", "confidence": 0.95}',
    'Example declaration: """John paid 20000""" -> {"memberId": null, "memberName": "", "amount": 20000, "recipientName": "", "confidence": 0.35}',
    'Example declaration: """Awa a versé vingt mille""" -> {"memberId": "m3", "memberName": "Awa Sarr", "amount": 20000, "recipientName": "", "confidence": 0.9}',
    'Example declaration: """I paid 20000 for Awa""" -> {"memberId": "m3", "memberName": "Awa Sarr", "amount": 20000, "recipientName": "", "confidence": 0.85}',
]) + "\n\nKnown members: {{members}}. Standard contribution {{standard}} FCFA, current recipient {{recipient}}. Declaration: \"\"\"{{declaration}}\"\"\""
INPUT_VARIABLES = ["members", "standard", "recipient", "declaration"]

result = create_managed_prompt(PROMPT_NAME, PROMPT_TEXT, INPUT_VARIABLES)
get_managed_prompt(PROMPT_NAME)

## Reuse it: invoke the managed prompt by ARN

This is the whole point — the saved prompt is not a trophy, it runs. Bedrock loads the prompt version directly when its ARN is passed as `modelId`, with `promptVariables` filling the template. This exact call shape is what the `parse-declaration` Lambda uses in production when `DECLARATION_PROMPT_ARN` is set.

In [14]:
PROMPT_ARN = "arn:aws:bedrock:us-east-1:587308451740:prompt/7OPO8WP4L5:1"

resp = bedrock.converse(
    modelId=PROMPT_ARN,
    promptVariables={
        "members": {"text": json.dumps([{"id": "m2", "name": "Moussa Ndiaye"}, {"id": "m4", "name": "Cheikh Fall"}])},
        "standard": {"text": "20000"},
        "recipient": {"text": "Cheikh Fall"},
        "declaration": {"text": "Moussa a payé 20000 pour Cheikh"}
    }
)
print(resp["output"]["message"]["content"][0]["text"])

```json
{"memberId": "m2", "memberName": "Moussa Ndiaye", "amount": 20000, "recipientName": "Cheikh Fall", "confidence": 0.95}
```


## What this proves

Prompt engineering with a closed loop: baseline, challenger, verdict, saved artifact — and the artifact runs in production behind one environment variable. Same model, same contract, strictly better compliance, reusable by the whole team.